# NIRNAY 450M quickstart

A 450M open-weight banking intent classifier: 87.9% on Banking77 test, fitted ECE 0.045, Apache-2.0. Built by [Eulogik](https://eulogik.com). Code: [github.com/eulogik/nirnay](https://github.com/eulogik/nirnay). Model: `eulogik/nirnay-450m` on Hugging Face.

Runs on CPU. No GPU needed.

No install needed: try the [live demo](https://huggingface.co/spaces/GautamKishore/nirnay-demo) first, type a message, get the intent.

Run below to do it locally.

In [ ]:
!pip install -q git+https://github.com/eulogik/nirnay

In [ ]:
from huggingface_hub import notebook_login, snapshot_download

notebook_login()  # paste a read token (model repo is private during verification)
local = snapshot_download("eulogik/nirnay-450m", allow_patterns=["phase_b.pt"])
ckpt = f"{local}/phase_b.pt"
print("checkpoint:", ckpt)

In [ ]:
from nirnay.agent import NirnayAgent

agent = NirnayAgent(device="cpu", checkpoint_path=ckpt, enable_byte_path=False)
print("loaded:", agent.report()["checkpoint_loaded"])

In [ ]:
messages = [
    "My card was charged twice for the same order.",
    "Where is my new card? It has been two weeks.",
    "I need to change my PIN before Friday.",
]
from nirnay.data import BANKING77_LABELS
criteria = {lab: lab.replace("_", " ") for lab in BANKING77_LABELS}
for m in messages:
    out = agent.system_one(m, {"intent": {
        "type": "choice",
        "instructions": "Classify the banking intent of the user message.",
        "criteria": criteria,
    }})
    ans = out["answers"]["intent"]
    top = sorted(ans["probabilities"].items(), key=lambda kv: kv[1], reverse=True)[:3]
    print(f"{m!r:60s} -> {top[0][0]} ({top[0][1]:.2f})")

In [ ]:
import matplotlib.pyplot as plt

out = agent.system_one(messages[0], {"intent": {
    "type": "choice",
    "instructions": "Classify the banking intent of the user message.",
    "criteria": criteria,
}})
probs = out["answers"]["intent"]["probabilities"]
top = sorted(probs.items(), key=lambda kv: kv[1], reverse=True)[:5]
plt.barh([k for k, _ in top][::-1], [v for _, v in top][::-1])
plt.xlabel("probability")
plt.title("Top 5 intents")
plt.tight_layout()
plt.show()

## Numbers (all measured, all reproducible)

| Model | Banking77 test (n=3080) | Setup |
|---|---|---|
| NIRNAY phase_b | 0.8792 | fine-tuned, this repo |
| Jev 1.13.0 | 0.803 (same 3,080 cases) | zero-shot API |

Eval harness: `scripts/eval_checkpoint.py` in the repo. Raw result JSONs ship under `eval/`.

Next: flip the repos public, live Space demo, GGUF/Ollama build.